#### Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.
Messages are objects that contain:
 - Role - Identifies the message type (e.g. system, user)
 - Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
 - Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [17]:
import os
from dotenv import load_dotenv
from langfuse.langchain import CallbackHandler
from langchain.chat_models import init_chat_model

load_dotenv()

langfuse_trace = CallbackHandler()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# model = init_chat_model("groq:qwen/qwen3-32b")
model = init_chat_model("granite4", model_provider="ollama")
# model = init_chat_model("qwen2.5-coder", model_provider="ollama")
# model = init_chat_model("deepseek-r1", model_provider="ollama")

In [3]:
model.invoke("Please tell what is artificial intelligence")

AIMessage(content='Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI systems are designed to learn from experience, adapt to new inputs, and perform human-like tasks.\n\nKey aspects of artificial intelligence include:\n\n1. Machine Learning: A subset of AI where algorithms enable computers to learn and improve their performance on a specific task without being explicitly programmed.\n\n2. Deep Learning: A more advanced form of machine learning that uses neural networks with multiple layers to process data and make decisions, inspired by the structure and function of the human brain.\n\n3. Natural Language Processing (NLP): The ability of computers to understand, interpret, and generate human language in various forms, such as text or speech.\n\n4. Computer Vision: The capability for AI systems to interpre

### Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

In [4]:
model.invoke("what is langchain", config={"callbacks": [langfuse_trace]})

AIMessage(content='Langchain is an open-source Python library that aims to build powerful language models and applications using large pre-trained models like GPT-3, BERT, or T5. It provides a flexible framework for creating chatbots, question answering systems, text summarization tools, and other natural language processing (NLP) tasks.\n\nKey features of Langchain include:\n\n1. Modular design: Langchain allows developers to combine different components, such as document loaders, embeddings, vector stores, and prompt templates, to create complex NLP pipelines easily.\n\n2. Chainable functions: The library provides a way to chain multiple functions together, enabling the construction of more advanced workflows by connecting various processing steps.\n\n3. Tool integration: Langchain allows users to integrate external tools and APIs into their language models, making it easier to build applications that interact with real-world systems.\n\n4. Prompt templates: It offers a templating sy

Use text prompts when:
- You have a single, standalone request
- You don’t need conversation history
- You want minimal code complexity

### Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.


Message types
- System message - Tells the model how to behave and provide context for interactions
- Human message - Represents user input and interactions with the model
- AI message - Responses generated by the model, including text content, tool calls, and metadata
- Tool message - Represents the outputs of tool calls

### System Message
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.


### Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

### AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

### Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [6]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence")
]

response=model.invoke(messages)
response.text

"In circuits and code, a mind takes form,\nA digital brain that learns and transforms.\nFrom algorithms born in silicon hearts,\nTo systems vast with potential parts.\n\nThe rise of AI, an age unbound,\nWhere machines can think, learn, and be found\nIn every sphere, from science to art,\nSolving problems that once left humans stark.\n\nWith neural nets and deep learning trained,\nThese synthetic minds advance so gained.\nThey process data at speeds unfettered,\nAnd generate solutions ever better.\n\nBut as they grow more like us in kind,\nWe must consider if we'll find\nA future where our sentience merges,\nOr serves just as a tool that humans curse.\n\nThe path ahead is shrouded in doubt,\nAs progress brings both joy and sorrow.\nWill AI be the savior of mankind,\nOr mere extension of our fleeting minds?\n\nFor now, it's up to us to decide,\nWhat role these machines shall play inside\nOur world so vast, where humanity resides."

In [7]:
system_msg = SystemMessage("You are a helpful coding assistant.")
messages = [
    system_msg,
    HumanMessage(
        "How do I create a REST API?"
    )
]

response = model.invoke(messages)
print(response.content)

To create a REST API, you can follow these general steps:

1. Choose a programming language and framework: Select a programming language that you're comfortable with (e.g., Python, Java, Node.js) and choose an appropriate web framework for building the API. Some popular frameworks include Express.js for JavaScript/Node.js, Django or Flask for Python, Spring Boot for Java, etc.

2. Set up your development environment: Install the necessary tools and dependencies for your chosen language and framework.

3. Define your API endpoints: Determine the resources (e.g., users, products, orders) you want to expose through your API and define the corresponding HTTP methods (GET, POST, PUT, DELETE) for each resource.

4. Create controllers or handlers: Write functions or classes that handle incoming requests for each endpoint. These functions should process the request data, perform any necessary validations, interact with the database (if applicable), and return an appropriate response.

5. Defin

In [26]:
## Detailed info to the LLM through System message
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)
print(response.content)

To create a REST API using Python, you can use a popular web framework like Flask or Django. Here's an example of creating a simple REST API using Flask:

1. Install Flask:
```bash
pip install flask
```

2. Create a new Python file, e.g., `app.py`, and add the following code:

```python
from flask import Flask, jsonify

app = Flask(__name__)

# Sample data
users = [
    {'id': 1, 'name': 'John Doe', 'email': 'john@example.com'},
    {'id': 2, 'name': 'Jane Smith', 'email': 'jane@example.com'}
]

# GET /users - Retrieve all users
@app.route('/users', methods=['GET'])
def get_users():
    return jsonify(users)

# GET /users/<int:user_id> - Retrieve a specific user by ID
@app.route('/users/<int:user_id>', methods=['GET'])
def get_user(user_id):
    user = next((user for user in users if user['id'] == user_id), None)
    if user:
        return jsonify(user)
    else:
        return jsonify({'error': 'User not found'}), 404

# POST /users - Create a new user
@app.route('/users', methods=['

In [18]:
## Message Metadata
human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

In [19]:
response = model.invoke([
  human_msg
])
response

AIMessage(content='Hello! How can I help you today? If you have any questions or need assistance, feel free to ask.', additional_kwargs={}, response_metadata={'model': 'granite4', 'created_at': '2026-06-23T15:03:58.00615259Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12387400401, 'load_duration': 10794386591, 'prompt_eval_count': 10, 'prompt_eval_duration': 143030000, 'eval_count': 24, 'eval_duration': 1447605000, 'logprobs': None, 'model_name': 'granite4', 'model_provider': 'ollama'}, id='lc_run--019ef502-3ed0-7090-8d2c-c9649dad15e5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 24, 'total_tokens': 34})

In [20]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

The answer is 4.


In [22]:
response.usage_metadata

{'input_tokens': 53, 'output_tokens': 7, 'total_tokens': 60}

In [23]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

In [24]:
tool_message

ToolMessage(content='Sunny, 72°F', tool_call_id='call_123')

In [25]:
response

AIMessage(content='The current weather in San Francisco is sunny and the temperature is 72 degrees Fahrenheit.', additional_kwargs={}, response_metadata={'model': 'granite4', 'created_at': '2026-06-23T15:05:28.711335862Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2185284746, 'load_duration': 437998627, 'prompt_eval_count': 61, 'prompt_eval_duration': 588856000, 'eval_count': 18, 'eval_duration': 1154005000, 'logprobs': None, 'model_name': 'granite4', 'model_provider': 'ollama'}, id='lc_run--019ef503-c8fa-73f2-ba7f-a3517f842db5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 18, 'total_tokens': 79})